In [6]:
# Cell 1 — install
!pip install torch torchvision matplotlib numpy -q

In [7]:
# Cell 2 — profiler
import torch
import torchvision.models as models
import time
import json

ENV_LABEL     = "colab_t4"   # change to "colab_l4" for L4 run
BATCH_SIZE    = 32
WARMUP_ITERS  = 5
MEASURE_ITERS = 20
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Environment : {ENV_LABEL}")
print(f"Device      : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Memory      : {props.total_memory/1024**3:.1f} GB")
    print(f"CUDA        : {torch.version.cuda}")

def count_flops(model_name):
    return {
        "ResNet18": 1_820_000_000,
        "ResNet50": 4_100_000_000,
        "VGG16":    15_500_000_000,
    }.get(model_name)

def measure_latency(model, dummy_input):
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(WARMUP_ITERS):
            model(dummy_input)
            torch.cuda.synchronize()
        for _ in range(MEASURE_ITERS):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_input)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000)
    mean_ms = sum(times) / len(times)
    std_ms  = (sum((t - mean_ms)**2 for t in times) / len(times)) ** 0.5
    return round(mean_ms, 3), round(std_ms, 3)

def profile_model(model_name, model, dummy_input):
    print(f"\nProfiling {model_name}...")
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    param_count = sum(p.numel() for p in model.parameters())
    param_MB    = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
    flops       = count_flops(model_name)
    mean_ms, std_ms = measure_latency(model, dummy_input)
    throughput  = round((BATCH_SIZE / mean_ms) * 1000, 1)
    allocated_MB = round(torch.cuda.memory_allocated() / 1024**2, 2)
    bw_bytes    = param_MB * 1024**2 * 2 + dummy_input[:1].numel() * dummy_input.element_size()
    ai          = round(flops / bw_bytes, 4)
    attainable  = round((flops / 1e9) / (mean_ms / 1000), 2)

    print(f"  Params    : {param_count:,} ({param_MB:.1f} MB)")
    print(f"  FLOPs     : {flops:,}")
    print(f"  Latency   : {mean_ms} ± {std_ms} ms")
    print(f"  Throughput: {throughput} samples/sec")
    print(f"  AI        : {ai} FLOPs/byte")
    print(f"  GFLOP/s   : {attainable}")
    print(f"  GPU Mem   : {allocated_MB} MB")

    return {
        "model":                   model_name,
        "param_count":             param_count,
        "param_MB":                round(param_MB, 2),
        "flops_per_sample":        flops,
        "latency_mean_ms":         mean_ms,
        "latency_std_ms":          std_ms,
        "throughput_samples_sec":  throughput,
        "gpu_memory_allocated_MB": allocated_MB,
        "bandwidth_bytes":         int(bw_bytes),
        "arithmetic_intensity":    ai,
        "attainable_gflops":       attainable,
    }

# Run
dummy_input = torch.randn(BATCH_SIZE, 3, 224, 224).to(DEVICE)
results = {
    "env_label": ENV_LABEL,
    "hardware": {
        "gpu_name":     torch.cuda.get_device_name(0),
        "gpu_memory_GB": round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1),
        "cuda_version": torch.version.cuda,
        "torch_version": torch.__version__,
    },
    "batch_size": BATCH_SIZE,
    "models": {}
}

for name, model in {
    "ResNet18": models.resnet18(weights=None).to(DEVICE),
    "ResNet50": models.resnet50(weights=None).to(DEVICE),
    "VGG16":    models.vgg16(weights=None).to(DEVICE),
}.items():
    results["models"][name] = profile_model(name, model, dummy_input)

with open(f"results_{ENV_LABEL}.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: results_{ENV_LABEL}.json")

# Also print nvidia-smi for screenshot
!nvidia-smi

Environment : colab_t4
Device      : cuda
GPU         : Tesla T4
Memory      : 14.6 GB
CUDA        : 12.8

Profiling ResNet18...
  Params    : 11,689,512 (44.6 MB)
  FLOPs     : 1,820,000,000
  Latency   : 28.888 ± 2.677 ms
  Throughput: 1107.7 samples/sec
  AI        : 19.3374 FLOPs/byte
  GFLOP/s   : 63.0
  GPU Mem   : 699.21 MB

Profiling ResNet50...
  Params    : 25,557,032 (97.5 MB)
  FLOPs     : 4,100,000,000
  Latency   : 94.446 ± 0.659 ms
  Throughput: 338.8 samples/sec
  AI        : 19.9943 FLOPs/byte
  GFLOP/s   : 43.41
  GPU Mem   : 699.21 MB

Profiling VGG16...
  Params    : 138,357,544 (527.8 MB)
  FLOPs     : 15,500,000,000
  Latency   : 162.075 ± 0.917 ms
  Throughput: 197.4 samples/sec
  AI        : 13.996 FLOPs/byte
  GFLOP/s   : 95.63
  GPU Mem   : 699.21 MB

Saved: results_colab_t4.json
Fri Mar 27 20:46:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.